# Data test — random HF replay → env + GUI

Pick a random game from `strakammm/generals_io_replays`, step it in the generals env, then open an interactive **pygame** replay (SPACE play/pause, ←/→ step, R restart, Q quit).


In [ ]:
from __future__ import annotations

import random
import sys
from pathlib import Path
from typing import Any

import jax.numpy as jnp
import numpy as np
from datasets import load_dataset

# Resolve repo root for `import generals`
_ROOT = Path.cwd().resolve()
if not (_ROOT / "generals").exists() and (_ROOT.parent / "generals").exists():
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from generals import create_action, create_initial_state, step, get_observation
from generals.core.action import compute_valid_move_mask
from generals.core.game import get_info
from generals.gui import ReplayGUI
from generals.gui.properties import GuiMode

SEED = None  # set e.g. 0 for a fixed random game; None = fresh each run
PAD_TO = 23
GUI_FPS = 8       # pygame replay speed
RUN_GUI = True    # set False to skip the window

print(f"generals import ok  (root={_ROOT})")


## 1. Load HuggingFace replays

In [ ]:
print("Loading dataset...")
dataset = load_dataset("strakammm/generals_io_replays")
train_dataset = dataset["train"]
print(f"Replays: {len(train_dataset)}")
print(f"Columns: {train_dataset.column_names}")


## 2. Pick a random game

In [ ]:
rng = random.Random(SEED)
idx = rng.randrange(len(train_dataset))
sample = train_dataset[idx]

print(f"random index: {idx}")
print(f"id:        {sample['id']}")
print(f"players:   {sample['usernames'][0]} vs {sample['usernames'][1]}")
print(f"stars:     {sample['stars']}")
print(f"map:       {sample['mapWidth']}×{sample['mapHeight']}")
print(f"moves:     {len(sample['moves'])}")
if sample["moves"]:
    print(f"last turn: {max(int(m[4]) for m in sample['moves'])}")


## 3. Convert HF replay → env grid + action sequence

Dataset move: `[player, start_tile, end_tile, is_50, turn]`  
Env action: `[pass, row, col, direction, split]` with dirs `0↑ 1↓ 2← 3→`.


In [ ]:
PASS = np.asarray(create_action(to_pass=True), dtype=np.int32)
DELTA_TO_DIR = {
    (-1, 0): 0,  # UP
    (1, 0): 1,   # DOWN
    (0, -1): 2,  # LEFT
    (0, 1): 3,   # RIGHT
}


def tile_to_rc(tile: int, width: int) -> tuple[int, int]:
    return divmod(int(tile), int(width))


def dataset_move_to_action(start_tile: int, end_tile: int, is_50: int, width: int) -> np.ndarray:
    sr, sc = tile_to_rc(start_tile, width)
    er, ec = tile_to_rc(end_tile, width)
    direction = DELTA_TO_DIR[(er - sr, ec - sc)]
    return np.array([0, sr, sc, direction, int(is_50)], dtype=np.int32)


def replay_to_grid(replay: dict[str, Any], pad_to: int = PAD_TO) -> np.ndarray:
    h, w = int(replay["mapHeight"]), int(replay["mapWidth"])
    if h > pad_to or w > pad_to:
        raise ValueError(f"map {(h, w)} exceeds pad_to={pad_to}")
    grid = np.zeros((h, w), dtype=np.int32)
    for tile in replay["mountains"]:
        r, c = tile_to_rc(tile, w)
        grid[r, c] = -2
    for tile, army in zip(replay["cities"], replay["cityArmies"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = int(army)
    for player, tile in enumerate(replay["generals"]):
        r, c = tile_to_rc(tile, w)
        grid[r, c] = player + 1  # 1 or 2
    padded = np.full((pad_to, pad_to), -2, dtype=np.int32)
    padded[:h, :w] = grid
    return padded


def replay_num_turns(replay: dict[str, Any]) -> int:
    moves = replay["moves"]
    if not moves:
        return 1
    return int(max(int(m[4]) for m in moves) + 1)


def replay_to_actions(replay: dict[str, Any], truncation: int) -> np.ndarray:
    """(T, 2, 5) — missing turns stay as pass."""
    w = int(replay["mapWidth"])
    seq = np.broadcast_to(np.stack([PASS, PASS]), (truncation, 2, 5)).copy()
    for move in replay["moves"]:
        player, start, end, is_50, turn = (int(x) for x in move)
        if turn >= truncation:
            continue
        seq[turn, player] = dataset_move_to_action(start, end, is_50, w)
    return seq


T = replay_num_turns(sample)
grid = replay_to_grid(sample)
actions = replay_to_actions(sample, truncation=T)
print(f"grid {grid.shape}  playable={np.count_nonzero(grid != -2)}")
print(f"actions {actions.shape}  non-pass={int(np.sum(actions[:, :, 0] == 0))}  T={T}")


## 4. Replay in the env

Silent step loop — BC pairs + GUI trajectory only. **No textual board printing.**


In [ ]:
DIR_NAME = {0: "↑", 1: "↓", 2: "←", 3: "→"}


def fmt_action(a: np.ndarray) -> str:
    a = np.asarray(a)
    if int(a[0]) == 1:
        return "pass"
    split = "50%" if int(a[4]) else "all-1"
    return f"({int(a[1])},{int(a[2])}){DIR_NAME[int(a[3])]} {split}"


DIR_NAME = {0: "↑", 1: "↓", 2: "←", 3: "→"}


def fmt_action(a: np.ndarray) -> str:
    a = np.asarray(a)
    if int(a[0]) == 1:
        return "pass"
    split = "50%" if int(a[4]) else "all-1"
    return f"({int(a[1])},{int(a[2])}){DIR_NAME[int(a[3])]} {split}"


h, w = int(sample["mapHeight"]), int(sample["mapWidth"])
state = create_initial_state(jnp.asarray(grid))

applied = 0
illegal = 0
bc_pairs: list[dict[str, Any]] = []
info = None
traj_states = [state]
traj_infos = [get_info(state)]

print(f"replaying id={sample['id']}  map={w}x{h}  T={T}  (silent — use GUI for board)")

for t in range(T):
    joint = actions[t]  # (2, 5)

    # Collect BC-style pairs: only the player who is actually moving.
    for p in range(2):
        a = np.asarray(joint[p])
        if int(a[0]) == 1:
            continue
        applied += 1
        o = get_observation(state, int(p))
        mask = compute_valid_move_mask(o.armies, o.owned_cells, o.mountains)
        legal = bool(mask[int(a[1]), int(a[2]), int(a[3])])
        if not legal:
            illegal += 1
        bc_pairs.append(
            {
                "turn": t,
                "player": p,
                "action": a.copy(),
                "obs": np.asarray(o.as_tensor()),
                "legal": legal,
            }
        )

    state, info = step(state, jnp.asarray(joint))
    traj_states.append(state)
    traj_infos.append(info)

    if bool(info.is_done):
        break

end_t = int(info.time) if info is not None else t
winner = int(info.winner) if info is not None else -1
print("--- replay summary ---")
print(f"ended at turn={end_t}  winner={winner} ({'P0' if winner == 0 else 'P1' if winner == 1 else 'none/unfinished'})")
print(f"non-pass moves applied={applied}  illegal_under_fog={illegal}")
print(f"BC pairs collected (mover obs→action)={len(bc_pairs)}")
print(f"trajectory frames for GUI: {len(traj_states)}")


## 5. Interactive GUI replay

Opens a pygame window over the recorded trajectory.

| Key | Action |
|-----|--------|
| SPACE | play / pause |
| ← / → (or H / L) | step frame |
| hold ← / → | scrub |
| R | restart |
| Q | quit |

Requires a local display (not Colab). Set `RUN_GUI = False` in the imports cell to skip.


In [ ]:
if not RUN_GUI:
    print("RUN_GUI=False — skipping pygame window")
else:
    agent_ids = [
        str(sample["usernames"][0]),
        str(sample["usernames"][1]),
    ]
    print(
        f"opening GUI  frames={len(traj_states)}  fps={GUI_FPS}  "
        f"{agent_ids[0]} vs {agent_ids[1]}"
    )
    print("controls: SPACE play/pause | ←/→ step | R restart | Q quit")
    gui = ReplayGUI(
        traj_states[0],
        agent_ids=agent_ids,
        mode=GuiMode.REPLAY,
        start_paused=True,
        fps=GUI_FPS,
    )
    gui.play(traj_states, traj_infos)
    print("GUI closed")


## 6. Sample BC pairs from this replay

Each pair is **only the moving player's** fogged `obs` → their `action`.


In [ ]:
show_n = min(8, len(bc_pairs))
print(f"showing {show_n}/{len(bc_pairs)} pairs\n")
for pair in bc_pairs[:show_n]:
    obs = pair["obs"]
    print(
        f"turn={pair['turn']:4d}  player={pair['player']}  "
        f"action={fmt_action(pair['action'])}  legal={pair['legal']}  "
        f"obs={obs.shape} dtype={obs.dtype}"
    )

if bc_pairs:
    # sanity: obs is (14, H, W) for that player only
    assert bc_pairs[0]["obs"].shape[0] == 14
    print("\nobs channel 0 (armies) nonzero cells:", int(np.count_nonzero(bc_pairs[0]["obs"][0])))
